# MS 3313 — Comprehensive Final Exam

## Modules 1–5 (R Basics → ML Basics → ANOVA/MANOVA → Dimension Reduction → KNN → Factor & Conjoint → Clustering → Discriminant Analysis)

**Course:** MS3313  
**Final:** Cumulative — covers every technical area in the lecture notebooks.

---

### Instructions

This notebook contains **eight questions**. For each question:

- Code cells already define the **dataset name**, **input variable name (X)**, and **output / model object name** you must use. **Do not rename them — graders look for these specific names.**
- Wherever you see `# YOUR CODE HERE — delete this comment when you add your code`, replace that line with your solution.
- Wherever you see `NULL    # replace NULL with your code`, replace `NULL` with your working expression.
- After each question, an **analysis answer** markdown cell is provided. Replace `*Your answer here*` with your written interpretation (3–6 sentences).
- All datasets come from the **`mlba`** package — no external CSVs needed.
- Use `set.seed(42)` whenever a partition / random start is required (already provided).

### Required object names

| Q | Module | Dataframe | Inputs (X) / Model | Output / Result |
|---|--------|-----------|--------------------|-----------------|
| 1 | M1 — ML basics | `air.df` | `X_air`, `train.df`, `valid.df`, `fare.lm` | `Y_fare_pred`, `rmse_fare`, `mae_fare`, `mape_fare` |
| 2 | M2 — ANOVA/MANOVA | `flights.df` | `aov_dep`, `tukey_dep`, `levene_dep`, `manova_flights` | (built-in objects) |
| 3 | M3 — Dimension reduction | `uni.df` | `pca_cov`, `pca_cor` | `n_pcs_kaiser` |
| 4 | M4.0 — KNN | `tayko.df` | `tayko_train`, `tayko_valid`, `train_X_tayko`, `valid_X_tayko`, `train_y_tayko`, `valid_y_tayko` | `acc_tayko`, `best_k_tayko`, `Y_tayko_pred`, `cm_tayko` |
| 5 | M4.1a — Factor analysis | `books.df` | `X_books_norm`, `fa_books` | `kmo_books`, `bartlett_books` |
| 6 | M4.1b — Conjoint | `profiles.df` | `cj_lm`, `partworth` | `attr_importance` |
| 7 | M5.0 — Clustering | `ewa.df` | `X_ewa_norm`, `d_ewa`, `hc_ewa`, `km_ewa` | `Y_ewa_hier`, `Y_ewa_km`, `ewa_profile` |
| 8 | M5.1 — LDA | `banks.df` | `X_banks`, `Y_banks`, `lda_banks` | `Y_banks_pred`, `cm_banks`, `accuracy_banks`, `new_bank`, `pred_new_bank` |

---


## Setup: Load Required Libraries

Run this cell as-is. Do **not** modify.

In [1]:
suppressPackageStartupMessages({
  library(mlba)
  library(dplyr)
  library(tidyr)
  library(tibble)
  library(ggplot2)
  library(caret)
  library(class)         # knn()
  library(MASS)          # lda()
  library(psych)         # KMO, cortest.bartlett, fa.parallel, fa
  library(GPArotation)
  library(car)           # leveneTest
  library(cluster)
  library(factoextra)
  library(conjoint)
  library(fastDummies)
  library(IRdisplay)
})

# dplyr::select wins after MASS is loaded
select <- dplyr::select

set.seed(42)
options(scipen = 999, digits = 4)


## Plot rendering — make charts legible

The default Jupyter R kernel often renders ggplot / `factoextra` figures
with a device that produces tiny or blurry axis text. **Every plot in this
exam must be drawn through a cairo PNG device** so axes, titles, and
legends are readable. The pattern is already provided in each plotting
cell — you only have to drop your plotting expression into the marked
spot:

```r
tmp_file <- tempfile(fileext = ".png")
png(tmp_file, width = 900, height = 600, res = 120,
    type = "cairo", family = "sans")

print(
  NULL    # replace NULL with your ggplot / fviz_* / plot(...) call
)

dev.off()
IRdisplay::display_png(file = tmp_file)
```

**Hints**

* For `print({ ... })` you may pass a single ggplot or `fviz_*` object.
* For base-graphics calls (e.g., `plot(hc); rect.hclust(...)`) put the
  drawing commands *between* `png(...)` and `dev.off()` — no `print()` needed.
* Keep `res = 120` and `family = "sans"` so the rendered fonts match the
  rest of the exam.


---
## Question 1 — Module 1: Multiple Linear Regression on Airfares

**Dataset:** `mlba::Airfares` (638 routes × 18 variables)

**Inputs (X):** all route variables after dropping the four ID/text columns
(`S_CODE`, `S_CITY`, `E_CODE`, `E_CITY`) and dummy-encoding the four categorical
predictors (`VACATION`, `SW`, `SLOT`, `GATE`).

**Output (Y):** `FARE` (average fare in dollars).

**Tasks:**
1. Load the data, drop the four ID columns, and dummy-encode the four categoricals → store the cleaned data in `air.df` and the predictor matrix in `X_air`
2. Partition 60% training / 40% validation with `set.seed(42)` → store in `train.df` and `valid.df`
3. Fit a multiple linear regression `FARE ~ .` on the training set → store in `fare.lm`
4. Predict on the validation set → store predictions in `Y_fare_pred`
5. Compute validation **RMSE**, **MAE**, and **MAPE** → store in `rmse_fare`, `mae_fare`, `mape_fare`
6. Identify the two predictors with the largest absolute *t*-statistic → store in `top2_air`
7. Provide a written interpretation in the **markdown answer cell** at the end of this question

### Step 1 — Load Data, Drop ID Columns, Dummy-Encode

Use `air.df` for the cleaned dataframe and `X_air` for the predictor matrix.

In [8]:
# Required objects: air.df, X_air

air.df <- mlba::Airfares %>%   
select(-S_CODE, -S_CITY, -E_CODE, -E_CITY) %>%
fastDummies::dummy_cols(select_columns = c("VACATION", "SW", "SLOT", "GATE"), remove_first_dummy = TRUE, remove_selected_columns = TRUE)

X_air  <- air.df %>% select(-FARE)

cat("Rows:", nrow(air.df), " Columns:", ncol(air.df), "\n")
str(air.df)


Rows: 638  Columns: 14 
'data.frame':	638 obs. of  14 variables:
 $ COUPON      : num  1 1.06 1.06 1.06 1.06 1.01 1.28 1.15 1.33 1.6 ...
 $ NEW         : int  3 3 3 3 3 3 3 3 3 2 ...
 $ HI          : num  5292 5419 9185 2657 2657 ...
 $ S_INCOME    : num  28637 26993 30124 29260 29260 ...
 $ E_INCOME    : num  21112 29838 29838 29838 29838 ...
 $ S_POP       : int  3036732 3532657 5787293 7830332 7830332 2230955 3036732 1440377 3770125 1694803 ...
 $ E_POP       : int  205711 7145897 7145897 7145897 7145897 7145897 7145897 7145897 7145897 7145897 ...
 $ DISTANCE    : int  312 576 364 612 612 309 1220 921 1249 964 ...
 $ PAX         : int  7864 8820 6452 25144 25144 13386 4625 5512 7811 4657 ...
 $ FARE        : num  64.1 174.5 207.8 85.5 85.5 ...
 $ VACATION_Yes: int  0 0 0 0 0 0 0 1 0 0 ...
 $ SW_Yes      : int  1 0 0 1 1 1 0 1 1 1 ...
 $ SLOT_Free   : int  1 1 1 0 1 1 1 1 1 1 ...
 $ GATE_Free   : int  1 1 1 1 1 1 1 1 1 1 ...


### Step 2 — 60/40 Train/Validation Partition

Store the partitions in `train.df` and `valid.df`.

In [ ]:
set.seed(42)
# YOUR CODE HERE — delete this comment when you add your code
# Required objects: train.df, valid.df


train_idx <- NULL    # replace NULL with your code
train.df  <- NULL    # replace NULL with your code
valid.df  <- NULL    # replace NULL with your code

cat("train rows:", nrow(train.df), " valid rows:", nrow(valid.df), "\n")


### Step 3 — Fit the Multiple Linear Regression

Fit `FARE ~ .` on the training set; store the model in `fare.lm`.

In [ ]:
# YOUR CODE HERE — delete this comment when you add your code
# Required object: fare.lm

fare.lm <- NULL    # replace NULL with your code

summary(fare.lm)


### Step 4 — Predict on the Validation Set

Store the predicted fares in `Y_fare_pred`.

In [ ]:
# YOUR CODE HERE — delete this comment when you add your code
# Required object: Y_fare_pred

Y_fare_pred <- NULL    # replace NULL with your code

head(Y_fare_pred)


### Step 5 — Compute Validation RMSE, MAE, and MAPE

Store the metrics in `rmse_fare`, `mae_fare`, `mape_fare`.

In [ ]:
# YOUR CODE HERE — delete this comment when you add your code
# Required objects: rmse_fare, mae_fare, mape_fare


rmse_fare <- NULL    # replace NULL with your code
mae_fare  <- NULL    # replace NULL with your code
mape_fare <- NULL    # replace NULL with your code  # express MAPE as a percentage

cat(sprintf("RMSE = %.3f\nMAE  = %.3f\nMAPE = %.3f%%\n",
            rmse_fare, mae_fare, mape_fare))


### Step 6 — Top 2 Most Influential Predictors (Largest |t|)

Store the two top rows of the coefficient table in `top2_air`.

In [ ]:
# YOUR CODE HERE — delete this comment when you add your code
# Required object: top2_air


top2_air <- NULL    # replace NULL with your code
top2_air


### Question 1 — Analysis Answer

*Your answer here* — In 3–6 sentences, interpret the results and give one business takeaway.

---


## Question 2 — Module 2: ANOVA & MANOVA on Flight Delays

**Dataset:** `mlba::FlightDelays` (2,201 flights × 13 variables)

**Tasks:**
1. Load the data, convert `CARRIER`, `DAY_WEEK`, and `Flight.Status` to factors → store in `flights.df`
2. Run a one-way **ANOVA**: does `CRS_DEP_TIME` differ across `CARRIER`? → store the model in `aov_dep`
3. Run **Tukey HSD** post-hoc on `aov_dep` → store in `tukey_dep`
4. Test homogeneity of variance with **Levene's test** (`car::leveneTest`) → store in `levene_dep`
5. Run a one-way **MANOVA**: do `CRS_DEP_TIME` and `DISTANCE` jointly differ across `DAY_WEEK`? → store in `manova_flights`; report **Pillai's trace**
6. Provide a written interpretation in the **markdown answer cell** at the end of this question

### Step 1 — Load Data and Set Factor Types

Store the cleaned dataframe in `flights.df`.

In [ ]:
# YOUR CODE HERE — delete this comment when you add your code
# Required object: flights.df

flights.df <- NULL    # replace NULL with your code

cat("n =", nrow(flights.df), "  carriers =", nlevels(flights.df$CARRIER), "\n")
table(flights.df$CARRIER)


### Step 2 — One-Way ANOVA: `CRS_DEP_TIME ~ CARRIER`

Store the ANOVA model in `aov_dep`.

In [ ]:
# YOUR CODE HERE — delete this comment when you add your code
# Required object: aov_dep

aov_dep <- NULL    # replace NULL with your code
summary(aov_dep)


### Step 3 — Tukey HSD Post-Hoc

Store the result in `tukey_dep` and show the five smallest adjusted *p*-values.

In [ ]:
# YOUR CODE HERE — delete this comment when you add your code
# Required object: tukey_dep

tukey_dep <- NULL    # replace NULL with your code

head(tukey_dep$CARRIER[order(tukey_dep$CARRIER[,"p adj"]), ], 5)


### Step 4 — Levene's Test for Homogeneity of Variance

Store the test result in `levene_dep`.

In [ ]:
# YOUR CODE HERE — delete this comment when you add your code
# Required object: levene_dep


levene_dep <- NULL    # replace NULL with your code
levene_dep


### Step 5 — One-Way MANOVA: `cbind(CRS_DEP_TIME, DISTANCE) ~ DAY_WEEK`

Store the MANOVA in `manova_flights` and report Pillai's trace.

In [ ]:
# YOUR CODE HERE — delete this comment when you add your code
# Required object: manova_flights

manova_flights <- NULL    # replace NULL with your code

summary(manova_flights, test = "Pillai")
summary.aov(manova_flights)   # univariate follow-ups


### Question 2 — Analysis Answer

*Your answer here* — In 3–6 sentences, interpret the results and give one business takeaway.

---


## Question 3 — Module 3: PCA on Universities

**Dataset:** `mlba::Universities` (1,302 universities × 20 variables)

**Inputs (X):** the 17 numeric metrics — drop `College.Name`, `State`, and the
binary public/private flag, then drop rows with any `NA`.

**Tasks:**
1. Load the data, drop the three non-numeric columns, drop rows with any `NA` → store in `uni.df`
2. Run PCA on the **covariance** matrix (raw scale) → store in `pca_cov`
3. Run PCA on the **correlation** matrix (`scale. = TRUE`) → store in `pca_cor`
4. Apply the **Kaiser rule** (eigenvalue ≥ 1) on `pca_cor` → store the count in `n_pcs_kaiser`; report the cumulative variance explained
5. Plot the **scree plot** and the **PC1–PC2 biplot** of `pca_cor`
6. Provide a written interpretation in the **markdown answer cell** at the end of this question

### Step 1 — Load and Clean the Data

Store the cleaned numeric dataframe in `uni.df`.

In [ ]:
# YOUR CODE HERE — delete this comment when you add your code
# Required object: uni.df


uni.df <- NULL    # replace NULL with your code

cat("n =", nrow(uni.df), "  p =", ncol(uni.df), "\n")


### Step 2 — PCA on the Covariance Matrix

Store in `pca_cov`. (Use `prcomp` with `scale. = FALSE`.)

In [ ]:
# YOUR CODE HERE — delete this comment when you add your code
# Required object: pca_cov

pca_cov <- NULL    # replace NULL with your code
round(summary(pca_cov)$importance[, 1:5], 4)


### Step 3 — PCA on the Correlation Matrix (Standardized)

Store in `pca_cor`. (Use `prcomp` with `scale. = TRUE`.)

In [ ]:
# YOUR CODE HERE — delete this comment when you add your code
# Required object: pca_cor

pca_cor <- NULL    # replace NULL with your code
round(summary(pca_cor)$importance[, 1:6], 4)


### Step 4 — Kaiser Rule

Store the number of components retained in `n_pcs_kaiser`.

In [ ]:
# YOUR CODE HERE — delete this comment when you add your code
# Required object: n_pcs_kaiser


n_pcs_kaiser <- NULL    # replace NULL with your code

eig <- pca_cor$sdev^2
cat("Kaiser keeps", n_pcs_kaiser, "PCs\n")
cat("Cumulative variance explained =",
    round(sum(eig[seq_len(n_pcs_kaiser)]) / sum(eig), 3), "\n")


### Step 5 — Scree Plot and PC1–PC2 Biplot

In [ ]:
tmp_file <- tempfile(fileext = ".png")
png(tmp_file, width = 900, height = 500, res = 120,
    type = "cairo", family = "sans")

# YOUR CODE HERE — delete this comment when you add your code
# Build a fviz_eig(...) scree plot inside print(...).

print(
  NULL    # replace NULL with your fviz_eig(pca_cor, ...) call
)

dev.off()
IRdisplay::display_png(file = tmp_file)


In [ ]:
tmp_file <- tempfile(fileext = ".png")
png(tmp_file, width = 900, height = 700, res = 120,
    type = "cairo", family = "sans")

# YOUR CODE HERE — delete this comment when you add your code
# Build a fviz_pca_biplot(...) plot of pca_cor inside print(...).

print(
  NULL    # replace NULL with your fviz_pca_biplot(pca_cor, ...) call
)

dev.off()
IRdisplay::display_png(file = tmp_file)


### Question 3 — Analysis Answer

*Your answer here* — In 3–6 sentences, interpret the results and give one business takeaway.

---


## Question 4 — Module 4.0: K-Nearest Neighbors on Tayko

**Dataset:** `mlba::Tayko` (2,000 catalog customers × 25 variables)

**Inputs (X):** all variables except `sequence_number` and `Spending`
(`Spending` is target leakage: `Spending > 0` ⇒ `Purchase = 1`).

**Output (Y):** `Purchase` as a factor with levels `c("No", "Yes")`.

**Tasks:**
1. Load the data, drop `sequence_number` and `Spending`, factor `Purchase` → store in `tayko.df`
2. Partition 60/40 with `set.seed(42)` → store in `tayko_train` and `tayko_valid`
3. Standardize using **training-set statistics only** (`caret::preProcess`) → store in `train_X_tayko`, `valid_X_tayko`, `train_y_tayko`, `valid_y_tayko`
4. Tune `k` over `1:15` using `class::knn` and validation accuracy → store the table in `acc_tayko` and the best `k` in `best_k_tayko`
5. Refit at `k = best_k_tayko`, store predictions in `Y_tayko_pred` and the confusion matrix in `cm_tayko`
6. Provide a written interpretation in the **markdown answer cell** at the end of this question

### Step 1 — Load and Clean the Data

Store in `tayko.df`. Make `Purchase` a factor with levels `c("No", "Yes")`.

In [ ]:
# YOUR CODE HERE — delete this comment when you add your code
# Required object: tayko.df

tayko.df <- NULL    # replace NULL with your code

cat("Rows:", nrow(tayko.df), "  Columns:", ncol(tayko.df), "\n")
table(tayko.df$Purchase)


### Step 2 — 60/40 Partition

Store in `tayko_train` and `tayko_valid`.

In [ ]:
set.seed(42)
# YOUR CODE HERE — delete this comment when you add your code
# Required objects: tayko_train, tayko_valid

train_idx    <- NULL    # replace NULL with your code
tayko_train  <- NULL    # replace NULL with your code
tayko_valid  <- NULL    # replace NULL with your code

cat("train:", nrow(tayko_train), "  valid:", nrow(tayko_valid), "\n")


### Step 3 — Standardize Predictors (Training Stats Only)

Store predictor matrices in `train_X_tayko`, `valid_X_tayko` and labels in
`train_y_tayko`, `valid_y_tayko`. Use `caret::preProcess(method = c("center","scale"))`.

In [ ]:
# YOUR CODE HERE — delete this comment when you add your code
# Required objects: pp_tayko, train_X_tayko, valid_X_tayko, train_y_tayko, valid_y_tayko

pp_tayko      <- NULL    # replace NULL with your code  (preProcess on training X)
train_X_tayko <- NULL    # replace NULL with your code
valid_X_tayko <- NULL    # replace NULL with your code
train_y_tayko <- NULL    # replace NULL with your code
valid_y_tayko <- NULL    # replace NULL with your code

cat("Train means (should be ~0):\n"); print(round(colMeans(train_X_tayko), 3))


### Step 4 — Tune `k` over 1..15

Store the per-k accuracy table in `acc_tayko` and the best `k` in `best_k_tayko`.

In [ ]:
# YOUR CODE HERE — delete this comment when you add your code
# Required objects: acc_tayko, best_k_tayko

acc_tayko    <- NULL    # replace NULL with your code  (data.frame with columns k, accuracy)
best_k_tayko <- NULL    # replace NULL with your code

print(acc_tayko)
cat("Best k =", best_k_tayko,
    "  validation accuracy =", round(max(acc_tayko$accuracy), 4), "\n")


### Step 5 — Final Confusion Matrix at `best_k_tayko`

Store predictions in `Y_tayko_pred` and the `caret::confusionMatrix` object in `cm_tayko`.

In [ ]:
# YOUR CODE HERE — delete this comment when you add your code
# Required objects: Y_tayko_pred, cm_tayko

Y_tayko_pred <- NULL    # replace NULL with your code
cm_tayko     <- NULL    # replace NULL with your code  (use positive = "Yes")

cm_tayko


### Question 4 — Analysis Answer

*Your answer here* — In 3–6 sentences, interpret the results and give one business takeaway.

---


## Question 5 — Module 4.1a: Factor Analysis on Charles Book Club

**Dataset:** `mlba::CharlesBookClub` (4,000 members × 24 variables)

**Inputs (X):** the 8 book-category purchase counts —
`ChildBks, YouthBks, CookBks, DoItYBks, RefBks, ArtBks, GeogBks, ItalCook`.

**Tasks:**
1. Subset to the 8 book-category columns → store in `books.df`
2. Standardize with `scale()` → store in `X_books_norm`
3. Verify factorability: **KMO** (`psych::KMO`) → store in `kmo_books`; **Bartlett's sphericity test** (`psych::cortest.bartlett`) → store in `bartlett_books`
4. Run a parallel-analysis scree plot (`fa.parallel`) to choose the number of factors
5. Run an **EFA with varimax rotation** (`psych::fa`) at `nfactors = 2` → store in `fa_books`; print loadings ≥ 0.30
6. Provide a written interpretation in the **markdown answer cell** at the end of this question

### Step 1 & 2 — Subset and Standardize

Store the 8-column dataframe in `books.df` and the standardized matrix in `X_books_norm`.

In [ ]:
# YOUR CODE HERE — delete this comment when you add your code
# Required objects: books.df, X_books_norm

book_vars <- c("ChildBks","YouthBks","CookBks","DoItYBks",
               "RefBks","ArtBks","GeogBks","ItalCook")

books.df     <- NULL    # replace NULL with your code
X_books_norm <- NULL    # replace NULL with your code

cat("n =", nrow(books.df), " variables =", ncol(books.df), "\n")


### Step 3 — KMO and Bartlett's Sphericity Test

Store in `kmo_books` and `bartlett_books`.

In [ ]:
# YOUR CODE HERE — delete this comment when you add your code
# Required objects: kmo_books, bartlett_books

kmo_books      <- NULL    # replace NULL with your code  (psych::KMO)
bartlett_books <- NULL    # replace NULL with your code  (psych::cortest.bartlett on cor(X_books_norm))

cat("Overall KMO:", round(kmo_books$MSA, 3), "\n")
print(bartlett_books)


### Step 4 — Parallel Analysis to Decide the Number of Factors

In [ ]:
tmp_file <- tempfile(fileext = ".png")
png(tmp_file, width = 800, height = 500, res = 120,
    type = "cairo", family = "sans")

# YOUR CODE HERE — delete this comment when you add your code
# Build the fa.parallel(...) call inside print(...). Use fa = "fa".

print(
  NULL    # replace NULL with your fa.parallel(X_books_norm, fa = "fa", ...) call
)

dev.off()
IRdisplay::display_png(file = tmp_file)


### Step 5 — EFA with Varimax Rotation, `nfactors = 2`

Store the result in `fa_books`. Print loadings with `cutoff = 0.30, sort = TRUE`.

In [ ]:
# YOUR CODE HERE — delete this comment when you add your code
# Required object: fa_books


fa_books <- NULL    # replace NULL with your code

print(fa_books$loadings, cutoff = 0.30, sort = TRUE)
fa_books$Vaccounted


### Question 5 — Analysis Answer

*Your answer here* — In 3–6 sentences, interpret the results and give one business takeaway.

---


## Question 6 — Module 4.1b: Conjoint Analysis (Streaming Service)

A streaming service tests three attributes:
`Price` (\$8 / \$12 / \$16), `Resolution` (HD / 4K), `Ads` (Yes / No).
A focus-group respondent rated each of the 12 profiles on a 1 (worst) – 9 (best)
scale.

**Tasks:**
1. Build the 3 × 2 × 2 = 12 profile design dataframe (with the ratings vector below) → store in `profiles.df`
2. Estimate part-worth utilities by fitting `Rating ~ Price + Resolution + Ads` → store in `cj_lm`
3. Compute the part-worths (level mean − grand mean) → store in `partworth` (a named list, one element per attribute)
4. Compute attribute importance % (range / sum of ranges × 100) → store in `attr_importance`
5. Identify the **best** and **worst** predicted profiles → store in `best_profile` and `worst_profile`
6. Provide a written interpretation in the **markdown answer cell** at the end of this question

### Step 1 — Build the Profile Design and Add Ratings

Store in `profiles.df`. Use the ratings vector exactly as provided.

In [ ]:
# YOUR CODE HERE — delete this comment when you add your code
# Required object: profiles.df

profiles.df <- expand.grid(
  Price      = factor(c("$8","$12","$16"), levels = c("$8","$12","$16")),
  Resolution = factor(c("HD","4K")),
  Ads        = factor(c("Yes","No"))
)
profiles.df$Rating <- c(7, 5, 3,  9, 6,
                        8, 5, 2,  9, 7,
                        6, 4)        # 12 ratings provided

profiles.df


### Step 2 — Linear Regression for Part-Worths

Store the model in `cj_lm`.

In [ ]:
# YOUR CODE HERE — delete this comment when you add your code
# Required object: cj_lm

cj_lm <- NULL    # replace NULL with your code

summary(cj_lm)


### Step 3 — Compute Part-Worths

Store in `partworth` as a named list (one element per attribute) where each
element is a vector of (level mean − grand mean).

In [ ]:
# YOUR CODE HERE — delete this comment when you add your code
# Required object: partworth


partworth <- list()
# YOUR CODE HERE — delete this comment when you add your code
# Fill in partworth$Price, partworth$Resolution, partworth$Ads

partworth$Price      <- NULL    # replace NULL with your code
partworth$Resolution <- NULL    # replace NULL with your code
partworth$Ads        <- NULL    # replace NULL with your code

partworth


### Step 4 — Attribute Importance (%)

Store in `attr_importance` (a named numeric vector summing to ~100).

In [ ]:
# YOUR CODE HERE — delete this comment when you add your code
# Required object: attr_importance


attr_importance <- NULL    # replace NULL with your code
round(attr_importance, 1)


### Step 5 — Best and Worst Predicted Profiles

Store in `best_profile` and `worst_profile`.

In [ ]:
# YOUR CODE HERE — delete this comment when you add your code
# Required objects: best_profile, worst_profile

profiles.df$Pred <- predict(cj_lm)
best_profile  <- NULL    # replace NULL with your code
worst_profile <- NULL    # replace NULL with your code

cat("BEST predicted profile:\n");  print(best_profile)
cat("\nWORST predicted profile:\n"); print(worst_profile)


### Question 6 — Analysis Answer

*Your answer here* — In 3–6 sentences, interpret the results and give one business takeaway.

---


## Question 7 — Module 5.0: Cluster Analysis on EastWestAirlines

**Dataset:** `mlba::EastWestAirlinesCluster` (3,999 frequent-flyers × 12 variables)

**Inputs (X):** the 10 behavioral variables — drop `ID.` and `Award.`.

**Tasks:**
1. Load the data, drop `ID.` and `Award.` → store in `ewa.df`; standardize → store in `X_ewa_norm`
2. Compute the Euclidean distance matrix → store in `d_ewa`
3. **Hierarchical clustering with Ward's linkage** (`method = "ward.D2"`) → store in `hc_ewa`; cut at `k = 4` → store labels in `Y_ewa_hier`
4. **K-means** with `centers = 4`, `nstart = 25`, `set.seed(42)` → store in `km_ewa`; store labels in `Y_ewa_km`
5. Cross-tabulate hierarchical vs k-means cluster labels
6. Profile the k-means clusters on the **original (unstandardized)** scale → store the table in `ewa_profile`
7. Plot the k-means cluster visualization (`fviz_cluster`)
8. Provide a written interpretation in the **markdown answer cell** at the end of this question

### Step 1 — Load, Drop ID/Award, Standardize

Store the cleaned dataframe in `ewa.df` and the standardized matrix in `X_ewa_norm`.

In [ ]:
# YOUR CODE HERE — delete this comment when you add your code
# Required objects: ewa.df, X_ewa_norm

ewa.df      <- NULL    # replace NULL with your code
X_ewa_norm  <- NULL    # replace NULL with your code

cat("n =", nrow(ewa.df), " p =", ncol(ewa.df), "\n")


### Step 2 — Euclidean Distance Matrix

Store in `d_ewa`.

In [ ]:
# YOUR CODE HERE — delete this comment when you add your code
# Required object: d_ewa

d_ewa <- NULL    # replace NULL with your code


### Step 3 — Hierarchical Clustering (Ward) and Cut at k = 4

Store the hclust object in `hc_ewa` and the cluster labels in `Y_ewa_hier`.

In [ ]:
# YOUR CODE HERE — delete this comment when you add your code
# Required objects: hc_ewa, Y_ewa_hier

hc_ewa     <- NULL    # replace NULL with your code   (method = "ward.D2")
Y_ewa_hier <- NULL    # replace NULL with your code   (cutree at k = 4)

table(Y_ewa_hier)


### Step 3b — Plot the Ward Dendrogram (with k = 4 boxes)

Use the cairo-PNG wrapper. Draw `plot(hc_ewa, labels = FALSE, ...)` and
overlay `rect.hclust(hc_ewa, k = 4, border = "red")`.

In [ ]:
tmp_file <- tempfile(fileext = ".png")
png(tmp_file, width = 1000, height = 600, res = 120,
    type = "cairo", family = "sans")

# YOUR CODE HERE — delete this comment when you add your code


NULL    # replace NULL with your plot(...) call
NULL    # replace NULL with your rect.hclust(...) call

dev.off()
IRdisplay::display_png(file = tmp_file)


### Step 4 — K-Means (k = 4)

Store the kmeans object in `km_ewa` and the labels in `Y_ewa_km`.

In [ ]:
set.seed(42)
# YOUR CODE HERE — delete this comment when you add your code
# Required objects: km_ewa, Y_ewa_km

km_ewa   <- NULL    # replace NULL with your code   (centers = 4, nstart = 25)
Y_ewa_km <- NULL    # replace NULL with your code

table(Y_ewa_km)


### Step 5 — Cross-Tabulate Hierarchical vs K-Means

In [ ]:
# YOUR CODE HERE — delete this comment when you add your code
# Print a cross-tabulation of Y_ewa_hier (rows) vs Y_ewa_km (columns).

NULL    # replace NULL with your code


### Step 6 — Cluster Profile (Original Scale)

Store the per-cluster mean table in `ewa_profile`.

In [ ]:
# YOUR CODE HERE — delete this comment when you add your code
# Required object: ewa_profile


ewa_profile <- NULL    # replace NULL with your code
ewa_profile


### Step 7 — K-Means Cluster Visualization

In [ ]:
tmp_file <- tempfile(fileext = ".png")
png(tmp_file, width = 900, height = 600, res = 120,
    type = "cairo", family = "sans")

# YOUR CODE HERE — delete this comment when you add your code
# Build the fviz_cluster(...) plot inside print(...).

print(
  NULL    # replace NULL with your fviz_cluster(km_ewa, data = X_ewa_norm, ...) call
)

dev.off()
IRdisplay::display_png(file = tmp_file)


### Question 7 — Analysis Answer

*Your answer here* — In 3–6 sentences, interpret the results and give one business takeaway.

---


## Question 8 — Module 5.1: Linear Discriminant Analysis on Banks

**Dataset:** `mlba::Banks` (20 banks × 5 variables)

**Inputs (X):** `TotCap.Assets`, `TotExp.Assets`, `TotLns.Lses.Assets`.

**Output (Y):** `Financial.Condition` recoded as a factor with labels
`c("Strong", "Weak")` (levels `0 = Strong, 1 = Weak`).

**Tasks:**
1. Load the data, drop `Obs`, recode `Financial.Condition` to a factor with labels `c("Strong","Weak")` → store in `banks.df`; create `X_banks` (3 predictors) and `Y_banks` (target factor)
2. Fit an **LDA**: `Financial.Condition ~ .` → store in `lda_banks`
3. Predict on the full data (n = 20 is too small to hold out) → store predicted classes in `Y_banks_pred`, the `caret::confusionMatrix` in `cm_banks`, and the overall accuracy in `accuracy_banks`
4. Predict the financial condition of a new bank with `TotCap.Assets = 8.0`, `TotExp.Assets = 0.11`, `TotLns.Lses.Assets = 0.55` → store the new observation in `new_bank` and the prediction in `pred_new_bank`
5. Provide a written interpretation in the **markdown answer cell** at the end of this question

### Step 1 — Load and Prepare Data

Store the cleaned dataframe in `banks.df`, predictors in `X_banks`, target in `Y_banks`.

In [ ]:
# YOUR CODE HERE — delete this comment when you add your code
# Required objects: banks.df, X_banks, Y_banks

banks.df <- NULL    # replace NULL with your code   (factor labels c("Strong","Weak"); drop Obs)
X_banks  <- NULL    # replace NULL with your code   (3 numeric predictors)
Y_banks  <- NULL    # replace NULL with your code   (target factor)

cat("Class balance:\n"); print(table(Y_banks))
banks.df


### Step 2 — Fit the LDA Model

Store in `lda_banks`.

In [ ]:
# YOUR CODE HERE — delete this comment when you add your code
# Required object: lda_banks

lda_banks <- NULL    # replace NULL with your code
lda_banks


### Step 3 — Confusion Matrix on the Full Data

Store predictions in `Y_banks_pred`, confusion matrix in `cm_banks`, accuracy in `accuracy_banks`.

In [ ]:
# YOUR CODE HERE — delete this comment when you add your code
# Required objects: Y_banks_pred, cm_banks, accuracy_banks

Y_banks_pred   <- NULL    # replace NULL with your code
cm_banks       <- NULL    # replace NULL with your code   (caret::confusionMatrix)
accuracy_banks <- NULL    # replace NULL with your code

cm_banks
cat("Overall accuracy:", round(accuracy_banks, 4), "\n")


### Step 4 — Predict for a New Bank

Store the new observation in `new_bank` and the prediction in `pred_new_bank`.

In [ ]:
# YOUR CODE HERE — delete this comment when you add your code
# Required objects: new_bank, pred_new_bank

new_bank      <- NULL    # replace NULL with your code   (data.frame with the 3 predictors)
pred_new_bank <- NULL    # replace NULL with your code   (predict(lda_banks, new_bank))

cat("Predicted class:", as.character(pred_new_bank$class), "\n")
cat("Posterior probabilities:\n"); print(round(pred_new_bank$posterior, 4))


### Question 8 — Analysis Answer

*Your answer here* — In 3–6 sentences, interpret the results and give one business takeaway.

---


## Submission Checklist

Before submitting, verify that:

- [ ] All `# YOUR CODE HERE — delete this comment when you add your code` comments have been **removed**
- [ ] All `NULL` placeholders have been **replaced** with working code
- [ ] The notebook **runs top-to-bottom without errors** (Kernel → Restart & Run All)
- [ ] All eight `*Your answer here*` markdown cells have been replaced with your written interpretation
- [ ] Variable / object names match the **Required object names** table at the top of the notebook
